# Synthetic Lygos ImageStack Workflow

This example builds a small toy time series with one steady source and one variable source, then uses Lygos's `ImageStack` methods for background subtraction, aperture photometry, and relative-variability mapping. The final figure shows the image-level inputs and the extracted pixel light curve.

The scene is synthetic and uses no TESS, Roman, or MAST data. It demonstrates the image workflow only; it is not a calibrated light curve or a mission performance result.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings(
    "ignore",
    message="Unable to find acceptable character detection dependency.*",
)

import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from IPython.display import Image, display

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    candidate for candidate in search_roots
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "lygos" / "imagestack.py").is_file()
)
for import_root in (
    repository_root,
    repository_root.parent / "miletos",
    repository_root.parent / "pergamon",
    repository_root.parent / "ephesos",
    repository_root.parent / "chalcedon",
    repository_root.parent / "tdpy",
):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from lygos.imagestack import ImageStack, tangent_plane_wcs
from lygos.variability import compute_variability_products, plot_variability_summary
from tdpy.verbosity import print

visuals_directory = repository_root / "examples" / "visuals"
visuals_directory.mkdir(parents=True, exist_ok=True)
random = np.random.default_rng(42)

In [ ]:
time_days = np.linspace(0.0, 3.0, 30)  # [day]
variability_period_days = 1.0  # [day]
exposure_seconds = 120.0  # [s]
rows, columns = np.indices((21, 21))
steady_star_rate = 10.0 + 100.0 * np.exp(
    -0.5 * ((columns - 10.0) ** 2 + (rows - 10.0) ** 2) / 1.5**2
)  # [e-/s]
image_count_frames = []
for epoch_days in time_days:
    variable_amplitude_rate = 60.0 * (
        1.0 + 0.5 * np.sin(2.0 * np.pi * epoch_days / variability_period_days)
    )  # [e-/s]
    variable_star_rate = variable_amplitude_rate * np.exp(
        -0.5 * ((columns - 4.0) ** 2 + (rows - 15.0) ** 2) / 1.5**2
    )  # [e-/s]
    expected_counts = (steady_star_rate + variable_star_rate) * exposure_seconds  # [electron]
    image_count_frames.append(random.poisson(expected_counts))
count_cube = np.asarray(image_count_frames, dtype=float)  # [electron]
flux_cube = count_cube / exposure_seconds  # [e-/s]
error_cube = np.sqrt(count_cube) / exposure_seconds  # [e-/s]
quality = np.zeros(time_days.size, dtype=int)
quality[3] = 1

image_stack = ImageStack(
    flux=flux_cube,
    time=time_days,
    error=error_cube,
    quality=quality,
    wcs=tangent_plane_wcs(
        SkyCoord(150.0, 2.0, unit="deg"), 21, 21.0  # [arcsec pixel^-1]
    ),
    unit="e-/s",
    time_format="BTJD",
    mission="TESS",
    label="Synthetic image cube",
)
print(f"Created {len(image_stack)} explicitly synthetic image frames.")

In [ ]:
processed_stack = image_stack.good().subtract_background()
threshold_aperture = processed_stack.threshold_aperture(threshold=5.0)
fixed_aperture = processed_stack.circular_aperture(3.0, center=(10.0, 10.0))
threshold_curve = processed_stack.aperture_photometry(threshold_aperture)
fixed_curve = processed_stack.aperture_photometry(fixed_aperture)
variability_products = compute_variability_products(processed_stack)
print(
    f"Good frames: {len(processed_stack)}; threshold-aperture pixels: "
    f"{int(threshold_aperture.sum())}; central circular-aperture pixels: "
    f"{int(fixed_aperture.sum())}."
)

comparison_figure, comparison_axis = plt.subplots(figsize=(8.0, 3.6), constrained_layout=True)
for curve, label, color in (
    (threshold_curve, "Threshold aperture", "#A51417"),
    (fixed_curve, "Fixed circular aperture", "#16697A"),
):
    normalized_flux = curve["flux"] / np.nanmedian(curve["flux"])
    comparison_axis.plot(
        curve["time"],
        normalized_flux,
        ".-",
        markersize=3,
        linewidth=0.8,
        color=color,
        label=label,
    )
comparison_axis.set_xlabel("Time [BTJD day]")
comparison_axis.set_ylabel("Normalized aperture flux")
comparison_axis.legend(loc="upper right")
comparison_axis.grid(False)
comparison_path = visuals_directory / "lygos_synthetic_aperture_light_curves.png"
comparison_relative_path = Path(os.path.relpath(comparison_path, Path.cwd()))
print(f"Writing to {comparison_relative_path.as_posix()}...")
comparison_figure.savefig(
    comparison_relative_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
display(comparison_figure)
plt.close(comparison_figure)

summary_path = visuals_directory / "lygos_synthetic_variability_summary.png"
summary_relative_path = Path(os.path.relpath(summary_path, Path.cwd()))
print(f"Writing to {summary_relative_path.as_posix()}...")
plot_variability_summary(
    processed_stack,
    summary_relative_path,
    title="Synthetic Lygos ImageStack variability",
)
display(Image(filename=str(summary_path)))

In [ ]:
try:
    from IPython.display import IFrame

    bokeh_path = visuals_directory / "lygos_synthetic_variability_summary.html"
    bokeh_relative_path = Path(os.path.relpath(bokeh_path, Path.cwd()))
    bokeh_products = plot_variability_summary(
        processed_stack,
        bokeh_relative_path,
        title="Synthetic Lygos ImageStack variability",
        typefileplot="html",
    )
except ImportError:
    print("Install the optional Lygos Bokeh dependencies to create the HTML view.")
else:
    print(f"Interactive image, variability, and pixel-curve view saved to {bokeh_relative_path.as_posix()}.")
    display(IFrame(src=bokeh_relative_path.as_posix(), width="100%", height=430))